# AI应用基础 · 工单M3.2《产线实时缺陷检测系统》 — 教师参考版
第9周 | 参考实现可直接运行验证 | 答案请勿下发学生

In [1]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())

subprocess.run([sys.executable, "数据与工具/generate_data.py", "m3yolo"], check=True)
print("数据已生成: data/")

工作目录: /Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/01-AI应用基础


M3.2 -> data/yolo_dataset 60 张+标注
完成。数据输出目录: /Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/01-AI应用基础/data
数据已生成: data/


## 参考实现 `m32`（含依赖导入）

In [2]:
import csv
import json
import sys
import numpy as np
import numpy as np  # 通用科学计算


def m32():
    import cv2
    try:
        import onnxruntime as ort
    except ImportError:
        print("本机未装onnxruntime: 参考run_ultralytics方案或装onnxruntime")
        return
    sess = ort.InferenceSession("data/weights/yolov8n.onnx",
                                providers=["CPUExecutionProvider"])
    import glob
    total = {}
    for p in sorted(glob.glob("data/yolo_dataset/images/*.npy"))[:10]:
        img = np.load(p).astype(np.uint8)
        x = cv2.resize(img, (640, 640)).astype(np.float32) / 255.0
        out = sess.run(None, {"images": x.transpose(2, 0, 1)[None]})[0][0]
        cls = out[4:]                       # (80,8400)
        n = int((cls.max(0) > 0.4).sum())
        total[p] = n
        print(p.split("/")[-1], "conf>0.4 检出:", n)
    print("阈值实验: 降低conf检出↑误检↑; 产线建议conf=0.4~0.6,"
          "imgsz=320在CPU上平衡精度与FPS。")


## 学生版TODO → 教师答案对照
学生版starter中的TODO与上方参考实现（或下方双解）的对应关系：

| TODO | 题面（学生看到的指导） |
|------|----------------------|
| `1` | TODO-1 解析: cls = out[4:]; conf=cls.max(0), |
|  | cid=cls.argmax(0); keep=conf>阈值 |
| `?` | （在此处编写你的代码——依据上方注释/文档字符串） |
| `2` | TODO-2: 循环调用runner, 统计20张图总检出数, 填入工单表格 |
| `3` | TODO-3: warmup 3帧后计时20帧, 打印平均FPS(只计推理段!) |

> 发放纪律：本对照表**勿随学生版下发**；课堂按层放提示（D层演示照做/C层只给思路行）。

## 执行与验证

In [3]:
m32()

img_000.npy conf>0.4 检出: 0
img_001.npy conf>0.4 检出: 0
img_002.npy conf>0.4 检出: 0
img_003.npy conf>0.4 检出: 0
img_004.npy conf>0.4 检出: 0


img_005.npy conf>0.4 检出: 0
img_006.npy conf>0.4 检出: 0
img_007.npy conf>0.4 检出: 0
img_008.npy conf>0.4 检出: 0
img_009.npy conf>0.4 检出: 0
阈值实验: 降低conf检出↑误检↑; 产线建议conf=0.4~0.6,imgsz=320在CPU上平衡精度与FPS。
